# Estatística 2 - Aula prática 2_3 em Python

## Unidade 2: Modelos de regularizacao ou penalidade   

## Secao 2.3: Regressao Lasso    

By Jose P. Leitão

In [14]:
import numpy as np
import pandas as pd
from scipy.stats import norm

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error

In [15]:
# Vamos utilizar a base de dados "wage" que eh um dataset
# que contem uma amostra com dados sobre a renda de
# familias
# Carregando o dataset
df = pd.read_csv('wage.csv')

In [16]:
df.head()

,husage,husearns,huseduc,husblck,hushisp,hushrs,kidge6,earns,age,black,educ,hispanic,union,kidlt6,hrwage
0,31,800,17,0,0,40,0,480.0,29,0,14,0,0,0,12.00000
1,33,950,13,0,0,60,0,455.0,30,0,12,0,0,1,11.37500
2,34,1000,14,0,0,50,1,102.0,31,0,12,0,0,0,5.10000
3,42,730,14,0,0,40,1,300.0,41,0,12,0,0,0,10.00000
4,45,1154,16,0,0,38,1,425.0,45,0,18,0,0,0,12.14286


In [21]:
y = df[['hrwage']].to_numpy() # converte para numpy
y = y.ravel() # garantir o shape devido a regressão Lasso aguardar y com o shape (n,) ao inves de (n,1)
X = df.drop(['hrwage'], axis=1)

In [22]:
# Vamos particionar o dataset em
# 80% para treinamento e 20% para teste
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [23]:
# Vamos checar as dimensoes das bases de treinamento e teste
print(f"Total de linhas e colunas do dataset : {df.shape} ")
print(f"Total de linhas e colunas do Treino : {X_treino.shape}")
print(f"Total de linhas e colunas do Teste : {X_teste.shape}")

Total de linhas e colunas do dataset : (1794, 15) 
Total de linhas e colunas do Treino : (1435, 14)
Total de linhas e colunas do Teste : (359, 14)


In [ ]:
# A base de treinamento possui 1435 linhas (observações) e 
# 14 colunas (variaveis)
# A base de teste possui 359 linhas (observações) e 14
# colunas (variaveis)

In [24]:
print(df.columns)

Index(['husage', 'husearns', 'huseduc', 'husblck', 'hushisp', 'hushrs',
       'kidge6', 'earns', 'age', 'black', 'educ', 'hispanic', 'union',
       'kidlt6', 'hrwage'],
      dtype='str')


In [25]:
# Vamos listas as colunas que necessitam padronizar (contínuas) e as que não devem ser padronizadas (binárias)
col_continuas = ['husage', 'husearns', 'huseduc', 'hushrs', 'earns', 'age', 'educ']
col_binarias = ['husblck', 'hushisp', 'kidge6', 'black', 'hispanic', 'union', 'kidlt6']

In [26]:
# Criar um preprocessamento para transformar apenas as variáveis contínuas e passar as binárias
processamento_X = ColumnTransformer(
    transformers=[
        ("cont", StandardScaler(), col_continuas),
        ("bin", "passthrough", col_binarias)
    ]
)


In [ ]:
#############################################################
#                     REGRESSAO LASSO                       #
#############################################################

In [27]:
# Pipeline: preprocessa e treina com a regressão Ridge
alphas = np.logspace(-4, 4, 100)

# Busca o melhor Alpha e avalia pelo R²
pipeline = Pipeline(steps=[
    ("preprocessamento", processamento_X),
    ("modelo", LassoCV(alphas=alphas, cv=5))
])

# Wrapper para transformar y
model = TransformedTargetRegressor(
    regressor=pipeline,
    transformer=StandardScaler()
)


model.fit(X_treino, y_treino)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",Pipeline(step... cv=5))])
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",StandardScaler()
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",None
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cont', ...), ('bin', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists 

In [ ]:
# Melhor valor de alfa
print(f"Melhor Alpha : {model.regressor_.named_steps['modelo'].alpha_ :.3f}")

Melhor Alpha : 0.002


In [43]:
# Lista os coeficientes

df_coef = pd.DataFrame({
    "variavel": X_treino.columns,
    "coeficiente": model.regressor_.named_steps['modelo'].coef_
})

print(df_coef)

    variavel  coeficiente
0     husage     0.023622
1   husearns     0.076596
2    huseduc     0.003406
3    husblck    -0.021398
4    hushisp     0.888507
5     hushrs     0.004213
6     kidge6     0.020426
7      earns     0.000000
8        age    -0.000000
9      black     0.037564
10      educ    -0.000000
11  hispanic    -0.000000
12     union     0.047335
13    kidlt6     0.086110


In [44]:
# testa a predição
y_pred_teste = model.predict(X_teste)

In [45]:
# Avaliação
rmse = root_mean_squared_error(y_teste, y_pred_teste)
mse = mean_squared_error(y_teste, y_pred_teste)
r2 = r2_score(y_teste, y_pred_teste)

print(f"RMSE no teste: {rmse:.2f}")
print(f"MSE no teste: {mse:.2f}")
print(f"R² no teste: {r2:.2f}")

RMSE no teste: 2.07
MSE no teste: 4.27
R² no teste: 0.85


In [46]:
# Vamos criar uma nova observação para prever o valor da hora salario
# husage = 40 anos (idade do marido)
# husearns = 551 (rendimento do marido em US$)
# huseduc = 13 (anos de estudo do marido)
# husblck = 0 (o marido nao eh preto)
# hushisp = 0 (o marido nao eh hispanico)
# hushrs = 40 (o marido trabalha 40 horas semanais)
# kidge6 = 0 (nao tem filhos maiores de 6 anos)
# earns = 355.5 (rendimento da esposa em US$)
# age = 37 anos (idade da esposa) 
# black = 0 (esposa nao eh preta)
# educ = 13 (esposa possui 13 anos de estudo)
# hispanic = 0 (esposa nao eh hispanica)
# union = 0 (o casal nao possui uniao registrada)
# kidlt6 = 0 (nao possui filhos com menos de 6 anos)
X_novo = pd.DataFrame({
    'husage': [40],
    'husearns': [551],
    'huseduc' : [13],
    'husblck': [0],
    'hushisp': [0],
    'hushrs': [40],
    'kidge6': [0],
    'earns': [355.5],
    'age': [37],
    'black': [0],
    'educ': [13],
    'hispanic': [0],
    'union': [0],
    'kidlt6': [0]  
})

In [47]:
# calcular a preidção
y_novo = model.predict(X_novo)[0]

print(f"Predição para o salario em horas é: {y_novo:.2f}")

Predição para o salario em horas é: 9.27


In [48]:
# O intervalo de confianca para o nosso exemplo eh:
n = len(y_treino) # tamanho da amostra de treino
m = y_novo # valor medio predito
s = y_treino.std()  # desvio padrao da amostra de treino
dam =  s/np.sqrt(n) # distribuicao da amostragem da media
CIlwr_ridge = m + (norm.ppf(0.025) * dam) # intervalo inferior
CIupr_ridge = m - (norm.ppf(0.025) * dam) # intervalo superior

# Os valores sao:
print(f" Intervalo de confiança inferior : {CIlwr_ridge:.2f}")
print(f" Intervalo de confiança superior : {CIupr_ridge:.2f}")


 Intervalo de confiança inferior : 8.98
 Intervalo de confiança superior : 9.56
